# Практическая работа № 6
## Паттерны Observer, Factory, Singleton

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр
**Модуль:** 6 — Паттерны: Observer, Factory, Singleton
**Продолжительность:** 4 ч аудиторно + 4 ч самостоятельно
**Формат сдачи:** Git-репозиторий + тесты + README + ADR
**Максимальный балл:** 5

---

## О чём эта работа

После модуля 5 проект «Тайна исчезнувшего дива» умеет управлять состояниями, поведением и командами, но:
- системы связаны напрямую;
- создание объектов зашито в `Game`;
- общие ресурсы (`AudioManager`, `Config`) не управляются.

В этой работе вы добавите три паттерна:
- **Observer** — EventBus для событий;
- **Factory** — фабрики врагов и предметов;
- **Singleton** — AudioManager и Config.

## Цели работы

После выполнения вы сможете:
- реализовывать Observer и EventBus (Pub-Sub);
- управлять подпиской и отпиской;
- создавать объекты через фабрики (Simple Factory, Factory Method);
- реализовывать потокобезопасный Singleton;
- обосновывать выбор каждого паттерна;
- тестировать подписку и создание объектов.

## План работы

| Шаг | Что делаем | Что изучаем |
|---|---|---|
| 0 | Настройка окружения | ветка Git |
| 1 | Аудит текущего кода | прямая связанность |
| 2 | Проектирование EventBus | типы событий |
| 3 | Реализация EventBus (Observer) | subscribe/unsubscribe/publish |
| 4 | Реализация фабрик | EnemyFactory, ItemFactory |
| 5 | Реализация Singleton | AudioManager, Config |
| 6 | Интеграция в Game | EventBus + Factory + Singleton |
| 7 | Юнит-тесты | подписка, создание, синглтон |
| 8 | Документация | ADR-006, UML |

**Важно:** этот блокнот содержит только задания. Готовые решения — в `module6_example.ipynb` (образец, не для копирования).

**Для углублённого изучения теории:**
- Nystrom R. «Game Programming Patterns»: [Observer](https://gameprogrammingpatterns.com/observer.html)
- [Refactoring Guru: Observer](https://refactoring.guru/design-patterns/observer), [Factory Method](https://refactoring.guru/design-patterns/factory-method), [Singleton](https://refactoring.guru/design-patterns/singleton)

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

for pkg in ["pytest", "pycodestyle"]:
    try:
        __import__(pkg)
        print(f"OK: {pkg}")
    except ImportError:
        print(f"MISSING: {pkg} — установите: pip install {pkg}")

---
# Шаг 0. Подготовка

```bash
git checkout main
git checkout -b module6-observer-factory-singleton
pytest -v
```

Все тесты из модулей 1–5 должны проходить.

---
# Шаг 1. Аудит: где прямая связанность

## Зачем этот шаг

Observer устраняет прямые вызовы между системами. Сначала нужно найти все места, где системы дёргают друг друга.

## Задание 1.1

Пройдитесь по системам (`MovementSystem`, `InvestigationSystem`, ...) и `Game`. Найдите:
- вызовы `ui.*`, `audio.*`, `achievements.*` внутри систем;
- создание врагов/предметов через `Enemy(...)` / `Item(...)` в `Game`;
- повторное создание `AudioManager` или `Config`.

Заполните таблицу:

| Место | Прямая зависимость | Что должно стать событием/фабрикой/синглтоном |
|---|---|---|
| ... | ... | ... |

**Сохраните в** `docs/analysis_module6.md`.

## Задание 1.2

Ответьте письменно:
1. Какие события происходят в игре? (не меньше 5)
2. Какие объекты создаются динамически? (враги, предметы)
3. Какие ресурсы должны быть в единственном экземпляре?

---
# Шаг 2. Проектирование EventBus

## Задание 2.1. Список событий

Составьте список событий игры. Минимум:
- `ClueFound` — найдена улика;
- `EnemyDefeated` — побеждён враг;
- `ItemPickedUp` — подобран предмет;
- `StateChanged` — смена состояния игры.

Опишите каждое событие: какие данные несёт (поля).

**Сохраните в** `docs/events_module6.md`.

## Задание 2.2. Ответьте

1. Кто издатель, кто подписчик для каждого события?
2. Нужен ли приоритет подписчиков? Где?
3. Как события пересекаются с `CommandHistory` из модуля 5?

---
# Шаг 3. Реализация EventBus (Observer)

## Задание 3.1

Создайте `core/events.py`:
1. Базовый класс события `Event` (или используйте dataclass-события).
2. `EventBus` с `subscribe`, `unsubscribe`, `publish`.
3. Обработку исключений в подписчиках (не прерывать цепочку).

**Требования:**
- `subscribe(event_type, handler)` добавляет обработчик.
- `unsubscribe(event_type, handler)` удаляет обработчик (не падает, если его нет).
- `publish(event)` вызывает обработчики типа события.

## Задание 3.2. Коммит

```bash
git add core/events.py
git commit -m "Add EventBus: Observer with subscribe/unsubscribe/publish"
```

---
# Шаг 4. Реализация фабрик

## Задание 4.1

Создайте `core/factories.py`:
1. `EnemySpec` (dataclass) — характеристики врага: `kind`, `hp`, `speed`, `damage`.
2. `ItemSpec` (dataclass) — `kind`, `name`, `description`.
3. `EnemyFactory` — `create(kind) -> Enemy`.
4. `ItemFactory` — `create(kind) -> Item`.

**Требования:**
- Фабрики читают характеристики из словаря `specs`.
- Неизвестный тип → понятное исключение (`KeyError` или своё).
- Новый тип объекта добавляется без правки `Game`.

## Задание 4.2. Коммит

```bash
git add core/factories.py
git commit -m "Add EnemyFactory and ItemFactory"
```

---
# Шаг 5. Реализация Singleton

## Задание 5.1

Создайте `core/singletons.py`:
1. `AudioManager` — Singleton с потокобезопасным `__new__`.
2. `Config` — Singleton с загрузкой настроек из словаря.

**Требования:**
- `AudioManager()` и `AudioManager()` возвращают один и тот же объект.
- Потокобезопасность через `threading.Lock` (двойная проверка).
- `Config` имеет `get(key, default)` и `set(key, value)`.

## Задание 5.2. Коммит

```bash
git add core/singletons.py
git commit -m "Add thread-safe Singleton: AudioManager, Config"
```

---
# Шаг 6. Интеграция в Game

## Задание 6.1

Обновите `Game`:
- `Game` создаёт `EventBus`, `EnemyFactory`, `ItemFactory`, `AudioManager`, `Config`.
- Системы публикуют события через `event_bus`, а не вызывают друг друга.
- Враги/предметы создаются через фабрики.
- `AudioManager` и `Config` — Singleton.

## Задание 6.2. Проверка

Запустите игру. Убедитесь, что:
- события доходят до подписчиков;
- враги и предметы создаются через фабрики;
- `AudioManager()` и `Config()` — один экземпляр.

## Задание 6.3. Коммит

```bash
git add game/
git commit -m "Integrate EventBus, Factory, Singleton into Game"
```

---
# Шаг 7. Юнит-тесты

## Задание 7.1. Тесты EventBus
`tests/test_events.py`:
- `test_subscribe_and_publish`;
- `test_unsubscribe`;
- `test_publish_to_multiple_subscribers`;
- `test_handler_exception_does_not_break_chain`.

## Задание 7.2. Тесты фабрик
`tests/test_factories.py`:
- `test_create_goblin`;
- `test_create_unknown_raises`;
- `test_specs_dataclass`.

## Задание 7.3. Тесты Singleton
`tests/test_singletons.py`:
- `test_audio_manager_singleton`;
- `test_config_singleton`;
- `test_config_get_set`.

## Задание 7.4. Запуск

```bash
pytest -v
```

Все тесты должны проходить.

---
# Шаг 8. Документация

## Задание 8.1. README
Обновите разделы:
- EventBus и события;
- фабрики врагов и предметов;
- Singleton для AudioManager и Config.

## Задание 8.2. ADR-006
`docs/adr_006_observer_factory_singleton.md`:
- Контекст: сильная связанность, создание в коде, дубли ресурсов.
- Решение: Observer, Factory, Singleton.
- Альтернативы: прямые вызовы, внедрение зависимостей, глобальные переменные.
- Последствия: плюсы и минусы.

## Задание 8.3. UML
Добавьте в `docs/uml_module6.md` диаграммы классов для каждого паттерна.

## Задание 8.4. PEP 8

```bash
pycodestyle core/ game/ tests/ main.py
```

Исправьте замечания.

## Задание 8.5. Финальный коммит

```bash
git add .
git commit -m "Document module 6: Observer, Factory, Singleton"
git checkout main
git merge module6-observer-factory-singleton
```

---
# Чек-лист
```
[ ] EventBus спроектирован
[ ] События описаны
[ ] subscribe/unsubscribe/publish работают
[ ] Исключения в подписчиках не ломают цепочку
[ ] EnemyFactory и ItemFactory реализованы
[ ] Новый тип объекта без правки Game
[ ] AudioManager и Config — Singleton
[ ] Singleton потокобезопасен
[ ] Game интегрирован
[ ] Все тесты проходят
[ ] README, ADR, UML обновлены
[ ] PEP 8 без замечаний
```

---

# Критерии оценки

| Критерий | Баллы |
|---|---:|
| Observer (EventBus) | 1.5 |
| Factory | 1.5 |
| Singleton | 1.0 |
| Интеграция | 0.5 |
| Тесты | 0.5 |
| **Итого** | **5.0** |

**Штрафы:** −1 за задержку, −2 за плагиат, −1 за отсутствие тестов.

**Бонусы (+1):** Abstract Factory, слабые ссылки в подписках, приоритеты подписчиков.

## Что дальше
В модуле 7 — архитектура игры: ECS, MVC, слоистая архитектура, базовые async/threading.